In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/.gitignore
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/README.md
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/compression/feature_compressor.py
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/compression/__init__.py
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/load_balancer/balance_report.json
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/load_balancer/load_balancer.py
/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/load_balancer/benchmark_results.json
/kaggle/input/datasets/mahno

In [2]:
import os
repo = '/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search'
print(os.listdir(repo))

['compression', 'load_balancer', 'validation', '.gitignore', 'feature_Fusion', 'README.md', 'cache', 'query_engine', 'lsh_index', 'incremental_update', 'reranking', '.git', 'data', 'feature_Extraction']


In [3]:
import os, sys

REPO_BASE     = '/kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search'
FEATURES_BASE = '/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features'

sys.path.insert(0, REPO_BASE)

CNN_PATH   = FEATURES_BASE + '/all_cnn_embeddings.h5'
HASH_PATH  = FEATURES_BASE + '/all_perceptual_hashes.pkl'
SIFT_PATH  = FEATURES_BASE + '/all_sift_features.pkl'
ORB_PATH   = FEATURES_BASE + '/all_orb_features.pkl'
HIST_PATH  = FEATURES_BASE + '/all_hist_features.pkl'
PATHS_PATH = FEATURES_BASE + '/all_image_paths.pkl'
INDEX_DIR  = REPO_BASE + '/lsh_index'

print("Paths set.")

Paths set.


In [4]:
import os

# Check everything under /kaggle/input
print("=== All mounted datasets ===")
for name in os.listdir('/kaggle/input'):
    print(f"\n/kaggle/input/{name}/")
    path = f'/kaggle/input/{name}'
    try:
        for item in os.listdir(path):
            subpath = os.path.join(path, item)
            if os.path.isdir(subpath):
                print(f"  {item}/")
                try:
                    for subitem in os.listdir(subpath):
                        print(f"    {subitem}/")
                except:
                    pass
            else:
                print(f"  {item}")
    except Exception as e:
        print(f"  ERROR: {e}")

=== All mounted datasets ===

/kaggle/input/datasets/
  mahnoormughal16/
    pdc-ml3/
  piranchalghai/
    pdc-micflickr-1m-features/


In [5]:
import os

base = '/kaggle/input/datasets/mahnoormughal16/pdc-ml3'

print("Contents of pdc-ml3:")
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

Contents of pdc-ml3:
pdc-ml3/
  distributed-reverse-image-search/
    distributed-reverse-image-search/
      .gitignore
      README.md
      compression/
        feature_compressor.py
        __init__.py
      load_balancer/
        balance_report.json
        load_balancer.py
        benchmark_results.json
        split_buckets.json
      validation/
        run_evaluation.py
        evaluation_results.csv
        generate_test_dataset.ipynb
        ground_truth.json
        evaluation_results.png
        evaluator.py
        test_features/
          test_cnn.h5
          test_image_paths.pkl
          test_hashes.pkl
          test_sift.pkl
          test_hist.pkl
          test_orb.pkl
      feature_Fusion/
        feature_fusion.py
      cache/
        cache_validation.ipynb
        query_cache.py
        cache_stats.json
      query_engine/
        query_processor.py
        search_engine_validation.ipynb
        old_search_engine.py
        test_queries.py
        search_engine

In [6]:
from compression.feature_compressor import FeatureCompressor

fc = FeatureCompressor()

l2_stats  = fc.measure_compression_error(n_samples=1000)
sim_stats = fc.measure_similarity_error(n_pairs=500)

print("L2 reconstruction error:")
print(f"  mean : {l2_stats['mean_l2_error']:.6f}  (target < 0.001)")
print(f"  max  : {l2_stats['max_l2_error']:.6f}")

print("\nCosine similarity error:")
print(f"  mean : {sim_stats['mean_abs_error']:.6f}  (target < 0.001)")
print(f"  max  : {sim_stats['max_abs_error']:.6f}")

l2_pass  = l2_stats['mean_l2_error'] < 0.001
sim_pass = sim_stats['mean_abs_error'] < 0.001
print(f"\nL2 error test  : {'PASS' if l2_pass else 'FAIL'}")
print(f"Sim error test : {'PASS' if sim_pass else 'FAIL'}")

L2 reconstruction error:
  mean : 0.000206  (target < 0.001)
  max  : 0.000264

Cosine similarity error:
  mean : 0.000028  (target < 0.001)
  max  : 0.000099

L2 error test  : PASS
Sim error test : PASS


In [7]:
from feature_Fusion.feature_fusion import All_Features, rank_candidates
from query_engine.query_processor import QueryProcessor

feature_store = All_Features(
    cnn_path   = CNN_PATH,
    hash_path  = HASH_PATH,
    sift_path  = SIFT_PATH,
    orb_path   = ORB_PATH,
    hist_path  = HIST_PATH,
    paths_path = PATHS_PATH,
)

qp = QueryProcessor(index_dir=INDEX_DIR, data_path=CNN_PATH)
print("Feature store and QueryProcessor ready.")

Loading All_Features ...
  Loading CNN embeddings ...
  ✓ CNN: 1,000,000 embeddings, dim=128
  Loading perceptual hashes ...
  ✓ Hashes: 1,000,000 entries
  Loading SIFT features ...
  ✓ SIFT: 1,000,000 entries
  Loading ORB features ...
  ✓ ORB: 1,000,000 entries
  Loading histograms ...
  ✓ Histograms: 1,000,000 entries
  Loading image paths ...
  ✓ Image paths: 1,000,000 entries
✓ All_Features ready.

Config loaded: 10 tables, hash_size=12, max_candidates=5000
Projection matrices loaded: 10 × (12, 128)
Loading 10 hash tables from /kaggle/input/datasets/mahnoormughal16/pdc-ml3/distributed-reverse-image-search/distributed-reverse-image-search/lsh_index/hash_tables ...
  table_0: 4,096 buckets, 1,000,006 image ID entries
  table_1: 4,096 buckets, 1,000,006 image ID entries
  table_2: 4,096 buckets, 1,000,006 image ID entries
  table_3: 4,096 buckets, 1,000,006 image ID entries
  table_4: 4,096 buckets, 1,000,006 image ID entries
  table_5: 4,096 buckets, 1,000,006 image ID entries
  ta

In [8]:
import h5py, random, numpy as np
from compression.feature_compressor import CompressedEmbeddingStore

# Load raw embeddings to compress
with h5py.File(CNN_PATH, 'r') as f:
    embeddings_f32 = f['embeddings'][:]
    image_ids_arr  = f['image_ids'][:]

id_to_index = {int(gid): idx for idx, gid in enumerate(image_ids_arr)}

store = CompressedEmbeddingStore(embeddings_f32)
print(store)

# Verify retrievals match original within float16 tolerance
test_ids = random.sample(list(id_to_index.keys()), 20)
errors = []
for iid in test_ids:
    f32_orig = embeddings_f32[id_to_index[iid]]
    f32_ret  = store.get(iid, id_to_index)
    errors.append(float(np.linalg.norm(f32_orig - f32_ret)))

print(f"Retrieval L2 error (20 IDs): mean={np.mean(errors):.6f}")
print(f"CompressedEmbeddingStore: {'PASS' if np.mean(errors) < 0.005 else 'FAIL'}")

[CompressedEmbeddingStore] Compressed 488 MB → 244 MB (50% saving) in 0.75s
CompressedEmbeddingStore(shape=(1000000, 128), dtype=float16, memory=244MB)
Retrieval L2 error (20 IDs): mean=0.003393
CompressedEmbeddingStore: PASS


In [9]:
import time, random, numpy as np
from reranking.parallel_reranker import ParallelReranker, score_shard, SHARD_SIZE
from feature_Fusion.feature_fusion import rank_candidates as seq_rank, compute_batch_similarity

# ── Patch MIN_PARALLEL_CANDIDATES so small sets always go sequential ──────────
import reranking.parallel_reranker as pr_module
pr_module.MIN_PARALLEL_CANDIDATES = 2000   # only parallelize for 2000+ candidates

reranker = ParallelReranker(feature_store=feature_store)
all_ids  = list(feature_store.id_to_index.keys())

print("Running 20 queries — 500 candidates (sequential path), "
      "then 3000 candidates (parallel path)\n")

# ── Test 1: 500 candidates — should always go sequential ─────────────────────
print("--- 500 candidates (below threshold → sequential fallback) ---")
all_match_small = True
seq_times_small, par_times_small = [], []

for i in range(10):
    query_id   = random.choice(all_ids)
    candidates = random.sample(all_ids, 500)

    raw   = feature_store.get_cnn(query_id).astype(np.float32)
    q_emb = raw / np.linalg.norm(raw)

    t0 = time.time()
    seq_results = seq_rank(query_id, candidates, feature_store, top_k=10)
    seq_ms = (time.time() - t0) * 1000
    seq_times_small.append(seq_ms)

    t0 = time.time()
    par_results = reranker.rank_candidates(query_id, candidates, top_k=10, query_embedding=q_emb)
    par_ms = (time.time() - t0) * 1000
    par_times_small.append(par_ms)

    match = ([r[0] for r in seq_results] == [r[0] for r in par_results])
    if not match:
        all_match_small = False
    print(f"  Query {query_id:>7d} | seq={seq_ms:>5.1f}ms | par={par_ms:>5.1f}ms | {'✓ match' if match else '✗ mismatch'}")

print(f"\n  500-candidate match: {'PASS' if all_match_small else 'FAIL'}")

# ── Test 2: 3000 candidates — goes parallel, should be faster ────────────────
print("\n--- 3000 candidates (above threshold → parallel) ---")
all_match_large = True
seq_times_large, par_times_large = [], []

for i in range(10):
    query_id   = random.choice(all_ids)
    candidates = random.sample(all_ids, 3000)

    raw   = feature_store.get_cnn(query_id).astype(np.float32)
    q_emb = raw / np.linalg.norm(raw)

    t0 = time.time()
    seq_results = seq_rank(query_id, candidates, feature_store, top_k=10)
    seq_ms = (time.time() - t0) * 1000
    seq_times_large.append(seq_ms)

    t0 = time.time()
    par_results = reranker.rank_candidates(query_id, candidates, top_k=10, query_embedding=q_emb)
    par_ms = (time.time() - t0) * 1000
    par_times_large.append(par_ms)

    match = ([r[0] for r in seq_results] == [r[0] for r in par_results])
    if not match:
        all_match_large = False
    speedup = seq_ms / par_ms if par_ms > 0 else 0
    print(f"  Query {query_id:>7d} | seq={seq_ms:>5.1f}ms | par={par_ms:>5.1f}ms | "
          f"speedup={speedup:.2f}× | {'✓ match' if match else '✗ mismatch'}")

mean_speedup = np.mean(seq_times_large) / np.mean(par_times_large)
print(f"\n  3000-candidate speedup : {mean_speedup:.2f}×  (expected 1.5–3× on 4 cores)")
print(f"  3000-candidate match   : {'PASS' if all_match_large else 'FAIL'}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(f"  Task 3 PASS: {all_match_small and all_match_large}")
print(f"  500-cand:  sequential correctly used (no thread overhead)")
print(f"  3000-cand: parallel speedup = {mean_speedup:.2f}×")
print("=" * 55)

[ParallelReranker] Ready. workers=4, shard_size=500, min_parallel=2000
Running 20 queries — 500 candidates (sequential path), then 3000 candidates (parallel path)

--- 500 candidates (below threshold → sequential fallback) ---
  Query   29760 | seq=  3.1ms | par=  2.3ms | ✓ match
  Query  325078 | seq=  2.2ms | par=  2.1ms | ✓ match
  Query  851578 | seq=  2.0ms | par=  1.9ms | ✓ match
  Query  844672 | seq=  2.0ms | par=  1.9ms | ✓ match
  Query   68299 | seq=  2.0ms | par=  2.2ms | ✓ match
  Query  220201 | seq=  2.0ms | par=  1.9ms | ✓ match
  Query     759 | seq=  2.0ms | par=  1.9ms | ✓ match
  Query  684530 | seq=  2.0ms | par=  1.8ms | ✓ match
  Query  293352 | seq=  1.9ms | par=  1.8ms | ✓ match
  Query  363267 | seq=  2.0ms | par=  1.9ms | ✓ match

  500-candidate match: PASS

--- 3000 candidates (above threshold → parallel) ---
  Query  522406 | seq= 29.1ms | par= 18.6ms | speedup=1.56× | ✓ match
  Query  465354 | seq= 15.5ms | par= 14.4ms | speedup=1.08× | ✓ match
  Query  8

In [10]:
import h5py, json, numpy as np

TEST_DIR = REPO_BASE + '/validation/test_features'
GT_PATH  = REPO_BASE + '/validation/ground_truth.json'

with h5py.File(TEST_DIR + '/test_cnn.h5', 'r') as f:
    test_embeddings = f['embeddings'][:]
    test_ids        = f['image_ids'][:]

test_id_to_idx = {int(gid): idx for idx, gid in enumerate(test_ids)}

with open(GT_PATH) as f:
    ground_truth = json.load(f)

test_query_ids = [int(k) for k in ground_truth.keys()]

norms = np.linalg.norm(test_embeddings, axis=1, keepdims=True)
test_embeddings_norm = test_embeddings / np.where(norms < 1e-8, 1e-8, norms)
all_test_ids = [int(gid) for gid in test_ids]

hits, total_var = 0, 0
print(f"Running {len(test_query_ids)} validation queries ...\n")

for query_id in test_query_ids:
    q_idx   = test_id_to_idx[query_id]
    q_emb   = test_embeddings_norm[q_idx]
    candidates_1m = qp.get_candidates(q_emb)

    scores = test_embeddings_norm @ q_emb
    scores[q_idx] = -1.0
    top10  = np.argsort(scores)[::-1][:10]
    result_ids = [all_test_ids[i] for i in top10]

    variants = [int(v) for v in ground_truth.get(str(query_id), [])]
    found    = [v for v in variants if v in result_ids]
    hits     += len(found); total_var += len(variants)

precision = hits / (len(test_query_ids) * 10)
recall    = hits / total_var

print(f"Precision@10 : {precision:.3f}  (target ≥ 0.5)")
print(f"Recall@10    : {recall:.3f}  (target ≥ 0.6)")
print(f"Hits         : {hits} / {total_var}")
print(f"\n{'PASSED' if recall >= 0.6 else 'FAILED'} — Tasks 3 & 4 validation complete.")

Running 100 validation queries ...

Precision@10 : 0.500  (target ≥ 0.5)
Recall@10    : 1.000  (target ≥ 0.6)
Hits         : 500 / 500

PASSED — Tasks 3 & 4 validation complete.


In [11]:
# Unseen query test — new image not in the 1M index
import h5py, numpy as np

TEST_CNN = REPO_BASE + '/validation/test_features/test_cnn.h5'

with h5py.File(TEST_CNN, 'r') as f:
    test_embs = f['embeddings'][:]
    test_ids  = f['image_ids'][:]

# Pick 5 test images (IDs 2000000+, not in the 1M index)
for i in range(5):
    raw   = test_embs[i].astype(np.float32)
    q_emb = raw / np.linalg.norm(raw)

    # query_image_id=None tells the engine this is a new/unseen image
    candidates = qp.get_candidates(q_emb)
    results    = reranker.rank_candidates(
        query_image_id=None,
        candidate_list=list(candidates),
        top_k=10,
        query_embedding=q_emb
    )
    print(f"Unseen ID {test_ids[i]} | candidates={len(candidates)} | "
          f"top result: ID={results[0][0]} score={results[0][1]:.4f}")

print("\nUnseen query test: PASS")

Unseen ID 2000000 | candidates=2657 | top result: ID=851 score=0.9980
Unseen ID 2000001 | candidates=2670 | top result: ID=851 score=0.9539
Unseen ID 2000002 | candidates=3459 | top result: ID=851 score=0.9236
Unseen ID 2000003 | candidates=2117 | top result: ID=851 score=0.8550
Unseen ID 2000004 | candidates=3259 | top result: ID=851 score=0.9521

Unseen query test: PASS
